# Analysis Notebook for Keithley CLI data

This notebook is used for analyzing the measurement data obtained from the Keithley CLI. It includes functions to download and combine `.txt` files from a Nextcloud share into a single DataFrame, as well as cells showing how to visualize the data using Plotly Express.

In [48]:
## Importing Libraries and Helper Functions
import plotly.express as px
import pandas as pd
from helpers.nextcloud import nextcloud_txt_to_dataframe, nextcloud_csv_to_dataframe

In [49]:
## Download and combine .txt files into a DataFrame, then print row counts per source file.
df = nextcloud_txt_to_dataframe()

# Ignore some of the file containing certain keywords, and plot the rest. For example, if the file name contains "_vdramp", we can assume it's Sternberg data and plot it separately.
ignore_keywords = [
    "dummy",
    "friday",
    "monday_pre",
    "empty",
]  # Add more keywords as needed

df = df[
    ~df["source_file"].str.contains("|".join(ignore_keywords), case=False, na=False)
]

df["source_file"] = (
    df["source_file"].str.split("/").str[-1]
)  # Keep only the filename, not the path

df["abs(Gate Current)"] = df["Gate Current"].abs()
df["abs(Drain Current)"] = df["Drain Current"].abs()
df["Flux"] = (
    df["Fluence"].diff() / df["Time"].diff()
)  # Calculate flux as the change in fluence over the change in time


print("RADEF txt files:")
for file in df["source_file"].unique() if not df.empty else []:
    subset = df[df["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

if df.empty:
    print("No data loaded from the Nextcloud share.")


RADEF txt files:
File: a31_000_gq.txt, Data Rows: 41
File: a31_000_idvd.txt, Data Rows: 111
File: a31_000_idvg.txt, Data Rows: 201
File: a31_001_vdramp.txt, Data Rows: 175
File: a32_000_gq.txt, Data Rows: 41
File: a32_000_idvd.txt, Data Rows: 111
File: a32_000_idvg.txt, Data Rows: 201
File: a32_001_gq.txt, Data Rows: 461
File: a32_001_idvd.txt, Data Rows: 111
File: a32_001_idvg.txt, Data Rows: 201
File: a32_001_irrad.txt, Data Rows: 254
File: a32_001b_gq.txt, Data Rows: 205
File: a32_002_irrad.txt, Data Rows: 173
File: a33_000_gq.txt, Data Rows: 40
File: a33_000_idvd.txt, Data Rows: 91
File: a33_000_idvg.txt, Data Rows: 201
File: a33_001_vdramp.txt, Data Rows: 195
File: a34_000_gq.txt, Data Rows: 38
File: a34_000_idvd.txt, Data Rows: 91
File: a34_000_idvg.txt, Data Rows: 201
File: a34_001_gq.txt, Data Rows: 458
File: a34_001_idvd.txt, Data Rows: 91
File: a34_001_idvg.txt, Data Rows: 201
File: a34_001_irrad.txt, Data Rows: 224
File: a34_001b_gq.txt, Data Rows: 192
File: a34_002_irrad.tx

## Gate Charge Data

In [50]:
# Get gate charge data and plot gate voltage vs. gate charge for files containing "_gq" in their name.

gate_charge_df = df[df["source_file"].str.contains("_gq", case=False, na=False)]

if gate_charge_df.empty:
    print("No gate charge data found in the DataFrame.")
else:
    # Integrate gate current over time to calculate gate charge (Q = \int_0^t I(t) * dt).
    dt = gate_charge_df["Time"].diff().fillna(0)  # Time intervals
    gate_charge_df["Gate Charge"] = (
        gate_charge_df["Gate Current"] * dt
    ).cumsum()  # Cumulative sum to get total charge

    fig = px.line(
        gate_charge_df, x="Gate Charge", y="Gate Voltage", color="source_file"
    )

    fig.update_layout(
        width=800,
        height=600,
        xaxis_range=[0, 50e-9],
    )
    fig.show()

## ID-VD Data

In [51]:
# Get ID-VD data and plot Drain Current and Gate Current vs. Drain Voltage on the same graph, with separate lines for each source file.
idvd_df = df[df["source_file"].str.contains("_idvd", case=False, na=False)]
if idvd_df.empty:
    print("No ID-VD data found in the DataFrame.")
else:
    fig_idvd = px.line(
        idvd_df,
        x="Drain Voltage",
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        symbol="variable",
    )
    fig_idvd.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvd.show()

## ID-VG Data

In [52]:
# Get ID-VG data and plot Drain Current and Gate Current vs. Gate Voltage on the same graph, with separate lines for each source file.
idvg_df = df[df["source_file"].str.contains("_idvg", case=False, na=False)]

if idvg_df.empty:
    print("No ID-VG data found in the DataFrame.")
else:
    fig_idvg = px.line(
        idvg_df,
        x="Gate Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        markers=True,
        symbol="variable",
    )

    fig_idvg.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvg.show()


## Sternberg Data

In [53]:
## Sternberg Data

# Ignore some of the file containing certain keywords, and plot the rest. For example, if the file name contains "_vdramp", we can assume it's Sternberg data and plot it separately.
ignore_keywords = ["dummy", "friday"]  # Add more keywords as needed

sternberg_df = df[df["source_file"].str.contains("_vdramp", case=False, na=False)]

if sternberg_df.empty:
    print("No Sternberg data found in the DataFrame.")
else:
    fig_sternberg = px.line(
        sternberg_df,
        x="Drain Voltage",
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        markers=True,
        symbol="variable",
        hover_data=["Time", "Fluence", "Flux"],
    )

    fig_sternberg.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_sternberg.show()

## Irradiation Data

In [54]:
# Get irradiation data and plot Drain Current and Gate Current vs. Fluence on the same graph, with separate lines for each source file.
irrad_df = df[df["source_file"].str.contains("_irrad", case=False, na=False)]

if irrad_df.empty:
    print("No irradiation data found in the DataFrame.")
else:
    fig_irrad = px.line(
        irrad_df,
        x="Fluence",
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        markers=True,
        symbol="variable",
        title="Irradiation Test Data",
    )

    fig_irrad.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_irrad.show()

## Sanity Check Between Vanderbilt and JYU Data

In [55]:
# Get Joshua's CSV files and plot them alongside RADEF's data for comparison.
df_joshua = nextcloud_csv_to_dataframe()

print("Joshua's CSV files:")
for file in df_joshua["source_file"].unique() if not df_joshua.empty else []:
    subset = df_joshua[df_joshua["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

idvg_df_joshua = df_joshua[
    df_joshua["source_file"].str.contains("_idvg", case=False, na=False)
]
# Remove DataName column if it exists, as it may interfere with plotting
idvg_df_joshua = idvg_df_joshua.drop(columns=["DataName"], errors="ignore")

# Rename joshua's ID-VG columns to match RADEF's for easier plotting
col_mapping = {
    "Vgate": "Gate Voltage",
    "Isource": "Source Voltage",
    "Idrain": "Drain Current",
    "Igate": "Gate Current",
}

# strip whitespace from column names in joshua's DataFrame before renaming
idvg_df_joshua.columns = idvg_df_joshua.columns.str.strip()

idvg_df_joshua = idvg_df_joshua.rename(columns=col_mapping)

# Combine both dataframes
idvg_combined = pd.concat([idvg_df, idvg_df_joshua], ignore_index=True)

fig_combined = px.line(
    idvg_combined,
    x="Gate Voltage",
    y=["Drain Current", "Gate Current"],
    color="source_file",
    symbol="variable",
    title="Combined ID-VG Characteristics (RADEF and Joshua's Data)",
)

fig_combined.update_layout(
    width=800,
    height=600,
    yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
)
fig_combined.show()


Joshua's CSV files:
File: raw_data/friday/A010_3_4_2026 9_19_32 AM_idvg.csv, Data Rows: 126
